# DIY Nifty Midcap 150 Momentum (`V3 / R3`) — Complete All-in-One Google Colab Workspace

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HarjasChadha/midcap-momentum-index/blob/main/notebooks/00_master_colab_all_in_one/Midcap_Momentum_Complete_Colab.ipynb)

This master notebook combines everything into one place for interactive research & verification in Google Colab — **while all original `.py` modules (`src/`, `scripts/`) and separate per-stage notebooks (`notebooks/01..09/`) remain cleanly separated.**

What you can inspect and run below:
1. **Colab / Local Environment Setup** (auto-clones `HarjasChadha/midcap-momentum-index`, installs `numba`, `scipy`, `yfinance`, `matplotlib`).
2. **Live Regime & Order Sheet (`R3` & `S3`)**: Inspect today's 4-Block Multi-Factor health scores, TrendGold `EMA150` gate (`GOLDBEES` vs `LIQUIDCASE`), and generate exact `Sequential-SlotCap` share orders for any custom capital (`₹1,00,000`, `₹5,00,000`, etc.).
3. **Interactive Equity & Drawdown Charts (`2018–2026`)**: Compare `R3`, `S3`, `R2`, `F2`, `Unhedged`, and `Midcap 150 EW` live inside Colab.
4. **16-Year Zero-Survivorship Numba + SciPy Proof (`2011–2026`, ETF-Free PIT)**: Review the `21,600+` Numba grid, Spearman OOS correlations, `[0.44, 0.10, 0.34, 0.12]` SciPy walk-forward weights, Gold/Liquid frontier, 2026 Gold Slump attribution, F&O comparison, and exact Python FIFO validation (`18.16%` `R3` vs `9.77%` passive EW).
5. **Re-run Live Strategy or Train/OOS Grid in Colab**: Interactive cells calling `src/` directly.

In [ ]:
# @title Environment & Repository Setup (Works in Google Colab & Local Jupyter)
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_ROOT = Path("/content/midcap-momentum-index")
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/HarjasChadha/midcap-momentum-index.git", str(REPO_ROOT)], check=True)
    os.chdir(REPO_ROOT)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yfinance", "beautifulsoup4", "pytz", "matplotlib", "numba", "scipy"], check=True)
else:
    p = Path.cwd().resolve()
    while p != p.parent and not (p / "src" / "regime.py").exists():
        p = p.parent
    REPO_ROOT = p
    os.chdir(REPO_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"Repo Root: {REPO_ROOT} | IN_COLAB={IN_COLAB}")

In [ ]:
# @title 1. Load Live State (`data/live_state.json`) & Inspect R3 Regime + TrendGold Gate
import json
import numpy as np
import pandas as pd

live = json.loads((REPO_ROOT / "data" / "live_state.json").read_text())
reg = live["regime"]
mf = reg["multifactor_blocks"]
tg = reg["trend_gold_status"]

print(f"As of: {live['as_of_date']} | Primary Strategy: {live['strategy_id']}")
print(f"Regime: {reg['state']} (Target Equity: {reg['target_equity_pct']}%) | Defensive ETF Gate: {reg['active_defensive_etf']}")
print(f"GOLDBEES: ₹{tg['gold_price']} vs EMA150: ₹{tg['gold_ema150']} ({tg['gold_vs_ema150_pct']:+.2f}%)")
print("\n4-Block Multi-Factor Regime Composite:")
for k in ["b1_trend", "b2_vol", "b3_breadth", "b4_macro_gold", "composite_raw", "composite_smooth"]:
    print(f"  {k:18s}: {mf[k]:+.3f}")

perf_df = pd.DataFrame(live["performance_summary"]).T[["cagr_pct", "max_dd_pct", "calmar", "sharpe", "sortino", "ann_turnover_pct", "avg_eq_pct"]]
print("\nVerified Live Universe Performance (2018-01-02 -> Current, Post-Cost & Post-Tax):")
display(perf_df)

In [ ]:
# @title 2. Plot Cumulative NAV & Drawdown Curves (R3 vs S3 vs F2 vs Unhedged vs Passive EW)
ch = live["charts"]
dates = pd.to_datetime(ch["dates"])
nav_df = pd.DataFrame(ch["nav"], index=dates)
dd_df = pd.DataFrame(ch["drawdown_pct"], index=dates)

try:
    import matplotlib.pyplot as plt
    cols = [
        ("R3_Risk9m_6M_MF4_TrendGold150", "R3 Optimal MF4 + TrendGold", "#16a34a", 2.4),
        ("S3_Sortino9m_6M_MF4_TrendGold150", "S3 Sortino MF4 + TrendGold", "#0891b2", 1.9),
        ("R2_Risk9m_6M_DD15_Sh75_GoldLiq50", "R2 Simple DD15", "#9333ea", 1.5),
        ("F2_9m_6M_DD12_Sh75_GoldLiq50", "F2 Legacy V1 DD12", "#4f46e5", 1.4),
        ("Unhedged_9m_6M", "Unhedged 9M", "#64748b", 1.2),
        ("Midcap150_EW_Universe", "Midcap 150 EW Benchmark", "#2563eb", 1.5),
    ]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
    for key, lbl, col, lw in cols:
        if key in nav_df:
            ax1.plot(nav_df.index, nav_df[key], label=lbl, color=col, linewidth=lw)
            ax2.plot(dd_df.index, dd_df[key], label=lbl, color=col, linewidth=lw)

    ax1.set_yscale("log")
    ax1.set_title("Cumulative Wealth (Log Scale, ₹1,00,000 Start, Post-Tax & 15bps Cost)")
    ax1.set_ylabel("NAV (₹)")
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc="upper left", ncol=2, fontsize=9)

    ax2.set_title("Underwater Drawdown (%)")
    ax2.set_ylabel("Drawdown %")
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Install matplotlib (`pip install matplotlib`) to plot interactive charts; tail NAV summary:")
    display(nav_df[["R3_Risk9m_6M_MF4_TrendGold150", "S3_Sortino9m_6M_MF4_TrendGold150", "F2_9m_6M_DD12_Sh75_GoldLiq50", "Midcap150_EW_Universe"]].tail(3))

In [ ]:
# @title 3. Custom Capital Calculator (Sequential-SlotCap + LIQUIDCASE Sweep)
CAPITAL_INR = 100_000  # @param {type:"number"}
PRESET = "R3"          # @param ["R3", "S3", "R2", "R7", "F2", "F4"]
SIMULATE_CRASH = False # @param {type:"boolean"}

from src.momentum import combine_sleeves, select_affordable_blend, select_sleeve_weights

u = pd.DataFrame(live["universe_scores"]).set_index("sym")
skey = "r12" if PRESET == "F4" else ("so9" if PRESET == "S3" else ("rk9" if PRESET in {"R3", "R2", "R7"} else "r9"))
wt_mode = "norm_z" if PRESET == "R7" else "equal"
sc = u[skey].dropna()
px = u["px"].dropna()

eq_frac = (0.50 if PRESET == "F4" else 0.25) if SIMULATE_CRASH else (reg["target_equity_pct"] / 100.0 if PRESET in {"R3", "S3"} else 1.0)
eq_budget = CAPITAL_INR * eq_frac
slv = [{"score_key": skey, "top_n": 20, "alloc": 0.5, "weighting": wt_mode}, {"score_key": skey, "top_n": 30, "alloc": 0.5, "weighting": wt_mode}]
w_map = select_affordable_blend({skey: sc}, slv, px, eq_budget) if CAPITAL_INR < 350_000 else combine_sleeves([(0.5, select_sleeve_weights(sc, 20, wt_mode)), (0.5, select_sleeve_weights(sc, 30, wt_mode))])

rows = []
eq_used = 0.0
for rk, (sym, w) in enumerate(w_map.items(), 1):
    p = float(px[sym])
    tgt = eq_budget * w
    sh = int(tgt // p)
    eq_used += sh * p
    rows.append({"Rank": rk, "Symbol": sym, "Weight%": round(w * eq_frac * 100, 2), "LTP": p, "Target₹": round(tgt, 2), "Shares": sh, "Actual₹": round(sh * p, 2)})

def_inr = CAPITAL_INR * (1.0 - eq_frac)
rem = max(0.0, CAPITAL_INR - eq_used)
liq_px = reg["etfs"]["LIQUIDCASE"]["price"]
print(f"Preset={PRESET} | Capital=₹{CAPITAL_INR:,.0f} | Eq Deployed=₹{eq_used:,.2f} ({len(rows)} slots) | Residual/Def Cash=₹{rem:,.2f} (~{int(rem//liq_px)} LIQUIDCASE @ ₹{liq_px})")
display(pd.DataFrame(rows))

In [ ]:
# @title 4. 16-Year Zero-Survivorship Audit, Numba Grid, Gold Frontier & F&O Tables
src16 = json.loads((REPO_ROOT / "reports" / "src_exact_16yr_verification.json").read_text())
print("=== 16-Year Exact Python FIFO Verification (Clean ETF-Free Semi-Annual PIT, 2011-2026) ===")
display(pd.DataFrame(src16["rows"]))
print("6-Phase Start-Date Rebalance Spread:", src16["phase_spread"])

r1 = json.loads((REPO_ROOT / "reports" / "numba_deep_research.json").read_text())
print("\n=== Gold / LIQUID_REALISTIC Defensive Frontier (With Sec 50AA 30% Liquid Tax vs 12.5% Gold LTCG) ===")
display(pd.DataFrame([x for x in r1["gold_liquid_frontier"] if x["signal"] == "EQ4_MultiFactor"]))

r2 = json.loads((REPO_ROOT / "reports" / "numba_part2_verification.json").read_text())
print("\n=== Crash Anatomy Including 2026 Gold Slump (-31.7%) ===")
display(pd.DataFrame(r2["crash_anatomy"]))
print("\n=== Realistic Monthly F&O Hedges (VIX-Scaled Theta, Apr-2026 STT & 30% F&O Tax) ===")
display(pd.DataFrame(r2["fno_hedges"]))

In [ ]:
# @title 5. Run Live Backtest Engine Interactively (`src/` Direct Execution)
from src.data_loader import fetch_and_cache_all
from src.momentum import compute_score_matrix
from src.regime import build_defensive_series, build_regime_features, compute_multifactor_state
from src.backtester import BacktestConfig, run_backtest

data = fetch_and_cache_all(REPO_ROOT, refresh=False)
adj_df, raw_df, bench_df = data["adj"], data["raw"], data["bench"]
feat = build_regime_features(adj_df, bench_df)
def_df = build_defensive_series(bench_df)
alloc_mf4, diag = compute_multifactor_state(feat, bench_df)
scores = {"risk_9m": compute_score_matrix(adj_df.ffill(limit=2), 189, 21, "risk_adj")}
slv = [{"score_key": "risk_9m", "top_n": 20, "alloc": 0.5, "weighting": "equal"}, {"score_key": "risk_9m", "top_n": 30, "alloc": 0.5, "weighting": "equal"}]

res_r3 = run_backtest(
    adj_df, raw_df, scores, def_df, alloc_mf4,
    BacktestConfig(start_date="2018-01-02", end_date=str(adj_df.index[-1]), initial_capital=100_000.0, rebalance_days=126, cost_bps_one_way=15.0, apply_tax=True, defensive_asset="TREND_GOLD150_OR_LIQ", trim_mode="weakest_first", sleeves=slv, discrete_shares=True, affordability_filter=True, sweep_idle_cash=True, sweep_asset="LIQUID_REALISTIC"),
    stock_ma50=adj_df.rolling(50, min_periods=30).mean(),
)
print("Live R3 ₹1L SlotCap Verified Metrics:")
print(json.dumps(res_r3["metrics"], indent=2))